# Mushroom Observation Data Pipeline - Kaggle Notebook

This notebook runs the **same pipeline as the command line**. It does not restate
the stage order or repeat any skip rules — it calls `run_pipeline.run_all()`, the
same entry point behind `python run_pipeline.py`, so both paths always behave
identically.

Every environmental layer is sampled from **Google Earth Engine** at the
observation points, so the run downloads no bulk rasters — no ERA5 netCDFs from
the CDS queue, no per-day CHIRPS GeoTIFFs, no multi-gigabyte WorldCover tiles,
and no SRTM DEM to process locally.

## Setup Instructions

### Before Running:
1. **Enable Internet** in Kaggle notebook settings (Settings → Internet → On)
2. **Add Secrets** (Kaggle Add-ons → Secrets):
   - `EARTHENGINE_PROJECT`: your Google Cloud project ID — the only credential
     the pipeline needs. Step 2 below prints where to find it if you are unsure.
   - Optional, fallback only: `OPENTOPOGRAPHY_API_KEY`, `CDSAPI_URL` / `CDSAPI_KEY`.
     Used only when Earth Engine is unavailable, or when `FETCH_RASTERS=1` asks
     for the source rasters on purpose.

### What comes from Earth Engine:

| Column | Dataset |
| --- | --- |
| `ndvi` | `COPERNICUS/S2_SR_HARMONIZED` |
| `soil_moisture` | `ECMWF/ERA5_LAND/DAILY_AGGR` |
| `prcp_d0..d6` | `UCSB-CHG/CHIRPS/DAILY` |
| `tmax_d0..d6`, `tmin_d0..d6` | `ECMWF/ERA5_LAND/DAILY_AGGR` |
| `land_cover` | `ESA/WorldCover/v200` |
| `elevation`, `slope`, `aspect` | `USGS/SRTMGL1_003` |
| `solar_exposure`, `wind_exposure`, `water_retention` | derived from the sampled terrain + `MERIT/Hydro/v1_0_1` |

### Data Storage:
Kaggle provides `/kaggle/working/` for output files that persist after the session.

## Step 1: Install Dependencies & Clone the Repo

In [ ]:
!pip install -q pyinaturalist earthengine-api requests numpy scipy pandas scikit-learn python-dotenv

# Fallback-only extras (bulk raster downloads, used when Earth Engine is
# unavailable or FETCH_RASTERS=1):
# !pip install -q cdsapi xarray netCDF4 rasterio rio-cogeo meteostat

In [ ]:
# Clone the repo into the Kaggle working directory. Safe to re-run: an existing
# checkout is updated rather than re-cloned.
import os
import subprocess
import sys

WORKING_DIR = '/kaggle/working'
REPO_DIR = os.path.join(WORKING_DIR, 'data-map')

if os.path.isdir(os.path.join(REPO_DIR, 'scripts')):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)
else:
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/skyfly200/data-map.git', REPO_DIR],
        check=True,
    )

# scripts/ holds the stage modules; the repo root holds utils/ and the data store.
for path in (os.path.join(REPO_DIR, 'scripts'), REPO_DIR):
    if path not in sys.path:
        sys.path.insert(0, path)
os.chdir(REPO_DIR)

print('Repo:', REPO_DIR)

## Step 2: Configure Environment

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# The pipeline defaults to <repo>/data for the per-species store, which is now
# the working directory, so DATA_DIR needs no override. Set it only to relocate
# the store (e.g. onto a Kaggle dataset mount).
# os.environ['DATA_DIR'] = '/kaggle/working/data'

user_secrets = UserSecretsClient()

# The only credential the Earth Engine path needs.
try:
    os.environ['EARTHENGINE_PROJECT'] = user_secrets.get_secret('EARTHENGINE_PROJECT')
    print("\u2713 Earth Engine project ID loaded")
except Exception as e:
    print(f"\u26a0 Earth Engine project not found: {e}")
    print("  Add it in Add-ons \u2192 Secrets \u2192 EARTHENGINE_PROJECT")

# Optional, fallback only: read when Earth Engine is unavailable, or when
# FETCH_RASTERS=1 asks for the source rasters on purpose.
try:
    os.environ['OPENTOPOGRAPHY_API_KEY'] = user_secrets.get_secret('OPENTOPOGRAPHY_API_KEY')
    print("\u2713 OpenTopography API key loaded (raster fallback)")
except Exception:
    print("\u2139 No OpenTopography key \u2014 not needed while Earth Engine is available")

try:
    cds_url = user_secrets.get_secret('CDSAPI_URL')
    cds_key = user_secrets.get_secret('CDSAPI_KEY')
    os.environ['CDSAPI_URL'] = cds_url
    os.environ['CDSAPI_KEY'] = cds_key
    with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
        f.write(f'url: {cds_url}\nkey: {cds_key}\n')
    print("\u2713 CDS API credentials loaded (raster fallback)")
except Exception:
    print("\u2139 No CDS credentials \u2014 not needed while Earth Engine is available")

# Which Google Cloud project will Earth Engine use? Prints where to find one if
# it is not configured yet.
import preflight
preflight.print_earthengine_project()

## Step 3: Authenticate Earth Engine

In [ ]:
import ee

# Kaggle has no browser, so authenticate in notebook mode if there is no stored
# credential yet. run_all() below will also initialise EE on its own.
try:
    ee.Initialize(project=os.environ.get('EARTHENGINE_PROJECT'))
    print("\u2713 Earth Engine initialized")
except Exception as e:
    print(f"Earth Engine not initialized yet ({e}); authenticating...")
    ee.Authenticate(auth_mode='notebook')
    ee.Initialize(project=os.environ.get('EARTHENGINE_PROJECT'))
    print("\u2713 Earth Engine initialized")

## Step 4: Run the Pipeline

One call, the same one `python run_pipeline.py` makes. It runs pre-flight, the
iNaturalist fetch, enrichment (Earth Engine first, cached rasters as fallback),
clustering, the GeoJSON export, and the coverage summary — applying the same skip
rules, so re-running only does what still needs doing.

Useful environment variables, all read identically from the CLI:

| Variable | Effect |
| --- | --- |
| `REFRESH_ALL=1` | re-run enrichment even if it previously completed |
| `SKIP_INAT_FETCH=1` | reuse the cached observations, skip the iNaturalist fetch |
| `USE_EARTH_ENGINE=0` | turn the Earth Engine stages off (raster fallback only) |
| `FETCH_RASTERS=1` | download the source rasters as well |
| `CLUSTER_COUNT=n` | number of environmental clusters |

In [ ]:
import run_pipeline

# The same entry point as `python run_pipeline.py`. It runs from the repo root
# (run_pipeline.ROOT_DIR, the checkout above) and restores this notebook's
# working directory when it finishes.
print('Pipeline root:', run_pipeline.ROOT_DIR)

run_pipeline.run_all()

## Step 5: Review Results

In [ ]:
import pandas as pd
import species_store as store

df = store.load_all(store.ENRICHED_DIR)
print(f"{len(df)} enriched observations, {df['species'].nunique()} species\n")

# Coverage per environmental column — how much Earth Engine actually filled in.
cols = ['ndvi', 'soil_moisture', 'land_cover', 'elevation', 'slope', 'aspect',
        'solar_exposure', 'wind_exposure', 'water_retention',
        'prcp_d0', 'prcp_d6', 'tmax_d0', 'tmin_d6', 'cluster']
present = [c for c in cols if c in df.columns]
coverage = pd.DataFrame({
    'filled': [df[c].notna().sum() for c in present],
    'missing': [df[c].isna().sum() for c in present],
    'coverage': [f"{df[c].notna().mean() * 100:.1f}%" for c in present],
}, index=present)
print(coverage.to_string())

In [ ]:
# Output files produced by the run
from pathlib import Path

for pattern in ('data/species/*.csv', 'data/enriched/*.csv',
                'public/data/*.geojson', 'public/data/*.json'):
    matches = sorted(Path(REPO_DIR).glob(pattern))
    total_mb = sum(m.stat().st_size for m in matches) / (1024 * 1024)
    print(f"{pattern:28s} {len(matches):4d} file(s)  {total_mb:7.2f} MB")

## Step 6: Download Results (Optional)

In [ ]:
# Zip the results for download
import shutil
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
archive_path = shutil.make_archive(
    os.path.join(WORKING_DIR, f'mushroom_data_{timestamp}'),
    'zip', REPO_DIR, base_dir='data',
)
print(f"\u2713 Created archive: {archive_path}")
print(f"  Size: {os.path.getsize(archive_path) / (1024*1024):.1f} MB")
print("\nDownload it from the notebook's Output tab.")

## Notes & Troubleshooting

### Where does `EARTHENGINE_PROJECT` come from?

It is the **project ID** of a Google Cloud project registered for Earth Engine —
not the display name and not the project number.

1. https://console.cloud.google.com/ — the project picker lists every project
   with its ID column. Create one if you have none.
2. https://code.earthengine.google.com/ — the Code Editor shows the active
   project top-right, and in the Assets tab.
3. Not registered yet? https://code.earthengine.google.com/register attaches a
   Cloud project to Earth Engine (free for noncommercial use).
4. Using gcloud already? `gcloud config get-value project`.

`preflight.print_earthengine_project()` (Step 2) prints the one this run will use.

### Common Issues

1. **Earth Engine authentication**: if `ee.Initialize` fails, re-run Step 3 —
   `ee.Authenticate(auth_mode='notebook')` works without a browser. Without a
   working session the pipeline falls back to bulk raster downloads, which need
   `OPENTOPOGRAPHY_API_KEY` plus CDS credentials and take dramatically longer.

2. **Earth Engine quotas**: sampling is batched by observation date — one request
   per date, not per observation — but a very wide date range can still hit the
   concurrent-request limit. Lower the parallelism:
   ```python
   import ee_enrich, species_store as store
   df = store.load_all(store.SPECIES_DIR)
   ee_enrich.enrich_precip_ee(df, max_workers=4)
   ```

3. **Memory limits**: Kaggle notebooks have ~16GB RAM. Reduce the species list,
   or set `SKIP_INAT_FETCH=1` to reuse what is already fetched.

4. **Session timeouts**: Kaggle sessions stop after ~12 hours. Every stage is
   resumable — re-run Step 4 and it continues where it left off.

### Resuming an interrupted run

Every enrichment stage only samples rows whose column is still empty, so re-running
Step 4 continues rather than starting over. The `.done` marker in `data/enriched/`
marks a finished run; `REFRESH_ALL=1` overrides it.

### Using the raster fallback deliberately

```python
os.environ['USE_EARTH_ENGINE'] = '0'
os.environ['FETCH_RASTERS'] = '1'
run_pipeline.run_all(root=WORKING_DIR)
```

### Next Steps

1. Upload `public/data/observations.geojson` to the Nuxt frontend's `public/data/`
2. Deploy to Netlify or serve statically
3. Or continue analysis here with the enriched CSVs